# Twisted Molybdenum Disulfide bilayers at various angles.

## 0. Introduction.

This notebook demonstrates how to generate a twisted interface between two materials using commensurate lattices. The example uses molybdenum disulfide (MoS2) as both the film and substrate materials. The notebook uses the new `create_commensurate_interface` function which first creates a slab from the material and then performs commensurate lattice matching to find valid supercells for the target twist angle. The algorithm searches for supercell matrices within specified size limits to achieve the target twist angle within tolerance. The generated interface is visualized and analyzed to determine the actual twist angle and the number of atoms in the interface.

> **Kaihui Liu, Liming Zhang, Ting Cao, Chenhao Jin, Diana Qiu, Qin Zhou, Alex Zettl, Peidong Yang, Steve G. Louie & Feng Wang**
> Evolution of interlayer coupling in twisted molybdenum disulfide bilayers. Nature Communications, 5, 4966. 2014.
 > [https://doi.org/10.1038/ncomms5966](https://doi.org/10.1038/ncomms5966)

The twisted MoS2 bilayers are shown in the following Figure 4 from the article.

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/interfaces/twisted-bilayer-molybdenum-disulfide/MoS2-twisted-bilayers.png" alt="Twisted MoS2 bilayers" width="600"/>

## 1. Prepare the Environment
### 1.1. Configurations from the article

The twist angles and interlayer distances below are the ones studied in the manuscript. The
distance is the article's Table S1 LDA value, the averaged **Mo–Mo** separation — not the
gap between the facing sulfur planes that the builder takes. Section 2.2 converts between the two.

At 0° and 60° the two layers are registered rather than twisted, and the article gives a
different distance for each registry: 6.1 Å for S over Mo (AA1/AB1), 6.2 Å for S over a
hollow site (AA2/AB2) and 6.8 Å for S over S (AA3/AB3). The builder has no registry parameter,
so the registry of each registered stack is measured in 2.2 rather than assumed.

The names below are how the [band structure notebook](interface_bilayer_twisted_commensurate_lattices_molybdenum_disulfide_SIMULATION.ipynb)
finds the materials created here, so changing one means changing it there too.

In [ ]:
# Uncomment a line to build that configuration as well. The three active by default are the ones
# the band structure notebook compares; the 13.2° and 46.8° cells hold 114 atoms and are slower.
INTERFACE_PARAMETERS = [
    {"name": "MoS2 bilayer 21.8deg d6.5", "angle": 21.8, "d_mo_mo": 6.5},
    {"name": "MoS2 bilayer AB1 d6.1", "angle": 60.0, "d_mo_mo": 6.1},
    {"name": "MoS2 bilayer AB1 d6.5", "angle": 60.0, "d_mo_mo": 6.5},
    # {"name": "MoS2 bilayer AA3 d6.8", "angle": 0.0, "d_mo_mo": 6.8},
    # {"name": "MoS2 bilayer 13.2deg d6.5", "angle": 13.2, "d_mo_mo": 6.5},
    # {"name": "MoS2 bilayer 38.2deg d6.5", "angle": 38.2, "d_mo_mo": 6.5},
    # {"name": "MoS2 bilayer 46.8deg d6.5", "angle": 46.8, "d_mo_mo": 6.5},
]

### 1.2. Set slab creation and search parameters

In [ ]:
# Slab creation parameters
MILLER_INDICES = (0, 0, 1)  # Miller indices for slab creation
NUMBER_OF_LAYERS = 1  # Number of layers in the slab

INTERFACE_VACUUM = 20.0  # in Angstroms

# Search algorithm parameters
MAX_REPETITION = None  # Maximum supercell matrix element value (None for automatic)
ANGLE_TOLERANCE = 0.5  # in degrees
RETURN_FIRST_MATCH = True  # If True, returns first solution within tolerance

# Visualization parameters
SHOW_INTERMEDIATE_STEPS = True
VISUALIZE_REPETITIONS = [3, 3, 1]

 ### 1.3. Install packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.4. Get input material
We'll use the MoS2 material from Standata.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

material = Material.create(Materials.get_by_name_and_categories("MoS2", "2D"))

print("Initial material properties:")
print(f"Formula: {material.formula}")
print(f"Number of atoms: {len(material.basis.elements.ids)}")

if SHOW_INTERMEDIATE_STEPS:
    visualize_materials(material, repetitions=VISUALIZE_REPETITIONS)
    visualize_materials(material, repetitions=VISUALIZE_REPETITIONS, rotation="-90x")

 ## 2. Generate Twisted Interfaces
 ### 2.1. Create the monolayer slab


In [ ]:
from mat3ra.made.tools.modify import translate_to_z_level
from mat3ra.esse.models.core.reusable.axis_enum import AxisEnum
from mat3ra.made.tools.helpers import create_slab
from mat3ra.made.tools.helpers import create_interface_commensurate as create_commensurate_interface


def cartesian_z(material, element=None):
    """z coordinates in Angstroms, optionally for a single element."""
    basis = material.basis.clone()
    basis.to_cartesian()
    return [z for (_, _, z), name in zip(basis.coordinates.values, basis.elements.values)
            if element is None or name == element]


slab = create_slab(
    crystal=material,
    miller_indices=MILLER_INDICES,
    number_of_layers=NUMBER_OF_LAYERS,
    vacuum=0.0, # No vacuum in the slab, it is a 2D material
)
slab = translate_to_z_level(slab, "center")

SLAB_THICKNESS = max(cartesian_z(slab, "S")) - min(cartesian_z(slab, "S"))
print(f"Monolayer S-S thickness: {SLAB_THICKNESS:.3f} Å")

visualize_materials(slab, rotation="-90x")

### 2.2. Create the twisted interfaces

The builder's `gap` is the separation between the facing sulfur planes, while the article tabulates
the Mo–Mo separation. The two differ by one monolayer's thickness, measured above rather than
taken as a constant, and the interface is measured again after it is built.

In [ ]:
import numpy as np


def mo_mo_separation(interface):
    """Separation of the two Mo planes, the quantity tabulated in the article."""
    z = sorted(cartesian_z(interface, "Mo"))
    half = len(z) // 2
    return sum(z[half:]) / half - sum(z[:half]) / half


def stacking_registry(interface):
    """Registry of a registered stack: S over Mo (AA1/AB1), over a hollow site (AA2/AB2)
    or over S (AA3/AB3). Each has its own interlayer distance in the article."""
    basis = interface.basis.clone()
    basis.to_crystal()
    coordinates = np.array(basis.coordinates.values)
    elements = np.array(basis.elements.values)
    in_plane_vectors = np.array(interface.lattice.vector_arrays)[:2, :2]
    lower = coordinates[:, 2] < coordinates[:, 2].mean()
    upper_sulfur = coordinates[~lower & (elements == "S")]
    facing = upper_sulfur[upper_sulfur[:, 2].argmin(), :2]

    def nearest(mask):
        delta = coordinates[mask][:, :2] - facing
        return np.linalg.norm((delta - np.round(delta)) @ in_plane_vectors, axis=1).min()

    if nearest(lower & (elements == "S")) < 0.25:
        return "AA3/AB3 (S over S)"
    if nearest(lower & (elements == "Mo")) < 0.25:
        return "AA1/AB1 (S over Mo)"
    return "AA2/AB2 (S over hollow)"


interfaces = []
for parameters in INTERFACE_PARAMETERS:
    interface = create_commensurate_interface(
        material=slab,
        target_angle=parameters["angle"],
        angle_tolerance=ANGLE_TOLERANCE,
        max_repetition_int=MAX_REPETITION,
        return_first_match=RETURN_FIRST_MATCH,
        direction=AxisEnum.z,
        gap=parameters["d_mo_mo"] - SLAB_THICKNESS,
        vacuum=INTERFACE_VACUUM,
    )
    interface.name = parameters["name"]
    # The commensurate cell is hexagonal, but the builder leaves the type generic; naming it here
    # is what lets a symbolic k-path resolve Γ, M and K downstream.
    interface.lattice.type = "HEX"
    interfaces.append(interface)
    registry = f", {stacking_registry(interface)}" if parameters["angle"] % 60 == 0 else ""
    print(f"{interface.name}: {parameters['angle']}°, {len(interface.basis.elements.ids)} atoms, "
          f"d(Mo-Mo) {mo_mo_separation(interface):.3f} Å "
          f"(target {parameters['d_mo_mo']} Å){registry}")

## 3. Preview the materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

for interface in interfaces:
    visualize_materials(interface, viewer="wave")

## 4. Save the materials

Each material is written to the `uploads` folder under its name, which is how the
[band structure notebook](interface_bilayer_twisted_commensurate_lattices_molybdenum_disulfide_SIMULATION.ipynb)
picks it up.

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials(interfaces)

for idx, interface in enumerate(interfaces):
    download_content_to_file(interface.to_json(), f"twisted_interface_{idx}.json")